# Data management

Post processing on TurboID pipeline results

In [ ]:
from pathlib import Path
import os
import polars as pl
import polars.selectors as cs
import pandas as pd
import requests
import math
import polars as pl
import os
import shutil
from unipressed import UniprotkbClient

# location of TurboID mass spec results
turboid_dir = Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/')
results_dir = turboid_dir / "03_results"

# experiments in unrelated cell types not included in manuscript
EXPERIMENTS_TO_DROP = ["EV2_45", "EV2_47", "CRW_A18", "EV2_28A", "EV2_28B", "EV1_100", "CRW_A26"]

# Supplementary data tables

In [ ]:
supplementary_data_dir = Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Supplementary Data/test/')
os.mkdir(supplementary_data_dir)

## Data S1: Experimental metadata and protein filtering metrics

Details of mouse biotin labeling experiments, mass spectrometry settings, and protein quantification statistics. An ROC filter was applied for tissue experiments, while a 15% enrichment filter was applied for serum experiments. B-cell, hepatocyte, and adipocyte enriched proteins were determined by retrieving genes from the human protein atlas in categories group enriched, cell type enriched, and cell type enhanced.

In [ ]:
metadata = pl.read_csv(
    results_dir
    / "02_censusout_processing_results/01_processed_files/conditions_metadata.csv"
)

# pull lists of enriched genes from data downloaded from human protin atlas
hpa_data = pl.read_csv(
    results_dir / "04_turboid_downstream_results/05_results/hpa_data_three_tissues.csv"
)
tissues = {"CD19": "B-cell", "alb": "Hepatocytes", "adip": "Adipocytes"}
hpa_enriched_proteins = {}
for tissue, cell_type in tissues.items():
    hpa_enriched_proteins[cell_type] = set(
        hpa_data.filter(pl.col("tissue") == tissue)["synonyms"]
    )


def label_tps(row, df, name):
    row[name] = len(df)
    annotation_categories = {
        "TP": "true_positive",
        "FP": "false_positive",
    }
    for annot in annotation_categories.keys():
        row[f"{name}_{annotation_categories[annot]}"] = len(
            df.with_columns(pl.col("annotation").fill_null("None")).filter(
                pl.col("annotation") == annot
            )
        )
    return row


def calc_percent(df, condition):
    return (len(df.filter(condition)) / len(df)) * 100



# Iterate through every experiment and label the number of true positives and false positives
# at each filtering step


rows = []
for row in metadata.iter_rows(named=True):
    exp_name = row["File name"].split("processed_census-out_")[1]
    row["experiment_name"] = exp_name

    raw_df = pl.read_csv(
        results_dir
        / f"04_turboid_downstream_results/01_raw_files_with_correct_channel_names_keratins_removed_TP_FP_annotated/{exp_name}.csv"
    )

    row = label_tps(row, raw_df, "number_proteins_unfiltered")

    cutoff_1_df = pl.read_csv(
        results_dir
        / f"04_turboid_downstream_results/02_filtering_per_cond_per_file/{exp_name}/passed_filter_processed_census-out_{exp_name}.csv"
    )

    row = label_tps(row, cutoff_1_df, "number_proteins_passing_intensity_and_cv_filter")

    suffix = ""
    if row["file_type"] == "tissue":
        folder = "01_tissue"
    else:
        folder = "02_serum"
        suffix = "_enrichment_filtered"

    cutoff_2_df = pl.read_csv(
        results_dir
        / f"04_turboid_downstream_results/05_results/{folder}/{exp_name}/final_protein_table_{exp_name}{suffix}.csv"
    )

    row = label_tps(row, cutoff_2_df, "number_proteins_passing_roc_or_enrichment_filter")

    row["percent_signalp"] = calc_percent(
        cutoff_2_df, pl.col("SignalP").eq(pl.lit("Y"))
    )
    row["percent_endoplasmic_reticulum_go_annotation"] = calc_percent(
        cutoff_2_df, pl.col("Gene Ontology (GO)").str.contains("GO:0005783")
    )

    for cell_type, enriched_genes in hpa_enriched_proteins.items():

        row[f"percent_{cell_type.lower()}_enriched_proteins_human_protein_atlas"] = (
            calc_percent(
                cutoff_2_df,
                pl.col("Symbol_human_jackson_homology_db").is_in(enriched_genes),
            )
        )

    rows.append(row)

df = pl.from_dicts(rows)

additional_metadata = pl.read_csv(
    turboid_dir / "02_metadata/metadata_turbo_id 2025_experiments_for_manuscript.csv"
)
df = df.join(other=additional_metadata, left_on="experiment_name", right_on="File name")

df = (
    df.with_columns(pl.col("Cre / Tissue").str.split(" ; "))
    .explode("Cre / Tissue")
    .with_columns(
        pl.col("Cre / Tissue").str.split(" / ").list.get(0).alias("cre"),
        pl.col("Cre / Tissue").str.split(" / ").list.get(1).alias("tissue"),
    )
    .group_by("experiment_name")
    .agg(pl.all().unique())
    .with_columns(
        pl.col(["cre", "tissue"])
        .list.eval(pl.element().cast(pl.String), parallel=True)
        .list.join(", ")
    )
    .select("cre", "tissue", "experiment_name")
    .join(other=df, how="right", on="experiment_name")
    .rename(
        {
            "experiment_name": "experiment",
            "Plex": "plex",
            "TurboID zygosity": "turboid_zygosity",
            "Diet duration": "diet_duration",
            "biotin labeling": "biotin_labeling",
            "Type": "type",
            "RTS?": "real_time_search",
        }
    )
)
display(df)

# change column ordering
s1_df = df.select(
    [
        "experiment",
        "cre",
        "tissue",
        "conditions",
        "plex",
        "diet_duration",
        "turboid_zygosity",
        "biotin_labeling",
        "type",
        "real_time_search",
    ]
    + [cs.contains("number")]
    + ["percent_signalp", "percent_endoplasmic_reticulum_go_annotation"]
    + [cs.contains("human")]
).unique()



s1_df.write_excel(
    workbook=supplementary_data_dir / "Data S1.xlsx",
    position=(2, 0),  # Start at row 2 (0-indexed), column 0
    freeze_panes=(3, 0),
    table_style=f"Table Style Light 8",
    column_widths=150,
)


## Data S2: Mean protein signal intensity across all experiments

Signal intensity data is mean of technical replicates. Median normalization was performed for cre+ channels for each condition. Cre- channels are unnormalized values from IP2 search. Published plasma concentrations are calculated from Michaud et al 2018.

In [ ]:
# compile protein metadata
# read protein metadata
# and add uniprot function information

fasta_table = pl.read_csv(
    results_dir
    / "/03_fasta_table_tp_fp/02_table_with_signal_p/main_fasta_table_with_signal_p.csv",
    ignore_errors=True,
)

def create_entry_cache(df):
    # Given a list of cysteine aggregation files,
    # create a cache of UniProt entries with each unique
    # protein.
    chunk_size = 500
    entry_dict = dict()
    uniprots = set()  # use set so each identifier is unique
    for uniprot in set(df["uniprot"]):
        if uniprot is not None and uniprot != "None":
            uniprots.add(uniprot)

    # Break the list of ids into smaller lists to not overwhelm uniprot
    i = 0
    uniprots = list(uniprots)  # convert to list so we can subscript
    chunks = [uniprots[x : x + chunk_size] for x in range(0, len(uniprots), chunk_size)]
    print("Querying UniProt...")
    for chunk in chunks:
        chunk_num = i * chunk_size
        print("Retrieved " + str(chunk_num) + " entries out of " + str(len(uniprots)))
        i = i + 1
        entries = UniprotkbClient.fetch_many(chunk)
        for entry in entries:
            entry_dict[entry["primaryAccession"]] = entry
    print("Done querying UniProt.")
    return entry_dict

cache = create_entry_cache(fasta_table.with_columns(pl.col("uniprot_id").alias("uniprot")))

df = fasta_table.to_pandas()

def get_function(x, cache):
    ret = set()
    if x in cache and "comments" in cache[x]:
        for i in cache[x]["comments"]:
            if "texts" in i and i["commentType"] == "FUNCTION":
                for j in i["texts"]:
                    ret.add(j["value"])
    return "|".join(list(ret))

df["uniprot_function"] = df["uniprot_id"].apply(
    get_function, cache=cache
)

df.to_csv(
    results_dir
    / "03_fasta_table_tp_fp/02_table_with_signal_p/main_fasta_table_with_signal_p_and_uniprot_function.csv",
)


fasta_table = pl.read_csv(results_dir
    / "03_fasta_table_tp_fp/02_table_with_signal_p/main_fasta_table_with_signal_p_and_uniprot_function.csv",)

In [ ]:
signal_intensity_dfs = []
for row in metadata.filter(pl.col("File name").str.contains_any(EXPERIMENTS_TO_DROP).not_()).unique().iter_rows(named=True):
    exp_name = row["File name"].split("processed_census-out_")[1]
    print(exp_name)
    suffix = ""
    if row["file_type"] == "tissue":
        folder = "01_tissue"
    else:
        folder = "02_serum"
        suffix = "_enrichment_filtered"

    signal_intensity_df = (
        (
            pl.read_csv(
                results_dir
                / f"04_turboid_downstream_results/05_results/{folder}/{exp_name}/final_protein_table_{exp_name}{suffix}.csv"
            )
            # text matching to get all signal intensity columns
            .select(
                "uniprot_id",
                cs.contains("cre")
                & ~cs.contains("ratio", "processed", "SecretomeP", "median"),
            )
        )
        .unpivot(
            index="uniprot_id",
        )
        # take mean signal intensity
        # by protein and condition
        .with_columns(
            pl.col("variable")
            .str.split("_")
            .list.slice(0, pl.col("variable").str.split("_").list.len() - 1)
            .list.join("_")
            .alias("condition")
        )
        .group_by(["uniprot_id", "condition"])
        .agg(pl.col("value").mean().alias("mean_signal_intensity"))
        .with_columns(pl.lit(exp_name).alias("experiment"))
    )

    signal_intensity_dfs.append(signal_intensity_df)

df = pl.concat(signal_intensity_dfs, how="vertical")

df = (
    s1_df.unique()
    .select(["experiment", "diet_duration", "type", "cre"])
    .with_columns(
        pl.when(pl.col("diet_duration") == "NA")
        .then(None)
        .otherwise(pl.col("diet_duration"))
        .alias("diet_duration")
    )
    .join(other=df, on="experiment", how="right")
    .with_columns(
        pl.concat_str(
            pl.col("cre"),
            pl.col("type"),
            pl.col("condition"),
            pl.col("diet_duration"),
            pl.col("experiment"),
            pl.lit("mean_signal_intensity"),
            separator="_",
            ignore_nulls=True,
        ).alias("experiment")
    )
    .pivot(on="experiment", values="mean_signal_intensity", index="uniprot_id")
)

aliases = pl.read_csv(
    "/Users/henrysanford/dev/turboid-figures/reference_data/protein_aliases.csv",
    truncate_ragged_lines=True,
)

published_concentrations = (
    pl.read_excel(
        turboid_dir
        / "../Reference datasets/Published data sets for comparison/Mouse/Absolute plasma concentrations/AbsolutePlasmaConcentrations_forHenry.xlsx"
    )
    .rename(
        {
            "Concentration in mouse plasma (pmol ml-1)": "published concentration in mouse plasma (pmol ml-1)",
            "Value sourced from":"mouse plasma concentration value sourced from"
        }
    )
    .with_columns(
        pl.col("published concentration in mouse plasma (pmol ml-1)").round(decimals=1)
    )
    .select(["uniprot_id", "published concentration in mouse plasma (pmol ml-1)", "mouse plasma concentration value sourced from"])
)

protein_metadata = (
    fasta_table.with_columns(
        pl.col("entry_name").str.split("_").list.get(0).alias("protein")
    )
    .join(other=aliases, how="left", on="protein")
    .join(other=published_concentrations, on="uniprot_id", how="left")
    .with_columns(pl.col("Mass").cast(pl.Int64))
    .select(
        [
            "uniprot_id",
            "entry_name",
            "Symbol_mouse_jackson_homology_db",
            "Symbol_human_jackson_homology_db",
            "alias",
            "Mass",
            "Outcyte",
            "Gene Ontology (GO)",
            "uniprot_function",
            "published concentration in mouse plasma (pmol ml-1)",
            "mouse plasma concentration value sourced from"
        ]
    )
)

s2_df = protein_metadata.join(other=df, on="uniprot_id", how="right")
s2_df.write_excel(
    workbook=supplementary_data_dir / f"Data S2.xlsx",
    position=(2, 0),  # Start at row 2 (0-indexed), column 0
    table_style=f"Table Style Light 8",
    column_widths=150,
    freeze_panes=(3,0),
    column_formats={"published concentration in mouse plasma (pmol ml-1)":"0.0"}
)

## Data S3: Differential expression of proteins in the mouse secretome

TMT-exp data showing protein expression changes in the mouse secretome. For every protein the log2 fold-change of the ratio between condtions was calculated. p-values were calculated with T-test for the means of two independent samples.

In [ ]:
mass_spec_folder = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/"
)
pipeline_folder = (
    mass_spec_folder / "03_results/04_turboid_downstream_results/05_results"
)
data = []
for folder in ["01_tissue", "02_serum"]:
    for experiment in os.listdir(pipeline_folder / folder):
        if experiment == ".DS_Store":
            continue

        suffix = ""
        if folder == "02_serum":
            suffix = "_enrichment_filtered"
        protein_table_path = pipeline_folder / (
            folder + "/" + experiment + "/final_protein_table_" + experiment + suffix + ".csv"
        )
        index_columns = [
            "entry_name",
            "uniprot_id",
            "uniprot_function",
            "pep_num",
            "annotation",
            "Mass",
            "Outcyte",
            "gene_name",
            "SWISS_PROT IDs_human_jackson_homology_db",
            "Gene Ontology (GO)",
        ]

        data.append(
            pl.scan_csv(protein_table_path)
            .unpivot(
                on=cs.contains(experiment), index=index_columns, variable_name="column"
            )
            .with_columns(
                pl.col("column")
                .str.replace("_processed_census-out", "")
                .str.split("_" + experiment + " - ")
            )
            .with_columns(
                pl.col("column").list.get(0).alias("variable"),
                pl.col("column")
                .list.get(1)
                .str.split(" (")
                .list.get(0)
                .alias("comparison"),
                pl.lit(experiment).alias("experiment"),
                pl.lit(folder.split("_")[1]).alias("sample_type"),
                pl.col("column")
                .list.get(1)
                .str.split(" (")
                .list.get(1)
                .str.split(" Proteins)")
                .list.get(0)
                .cast(pl.Int64)
                .alias("number_of_proteins"),
            )
            .drop("column")
            .collect()
            .pivot(on="variable", values="value")
        )

conditions_metadata = pl.read_csv(
    mass_spec_folder / "02_metadata/conditions_metadata.csv"
).with_columns(
    pl.col("File name").str.split("census-out_").list.get(1).alias("experiment")
)
de_results = (
    pl.concat(data, how="vertical_relaxed")
    .join(
        other=conditions_metadata.select(["experiment", "time_point"]), on="experiment"
    )
    .with_columns(pl.col("log2_FC").cast(pl.Float64))
).filter(
    pl.col("experiment")
    .str.contains_any(EXPERIMENTS_TO_DROP)
    .not_(),
    #pl.col("comparison").str.contains("cre").not_(),
)

more_metada = (
    pl.read_csv(
        "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/02_metadata/metadata_turbo_id 2025.csv"
    )
    .with_columns(
        pl.col("Cre / Tissue").str.split(" / ").list.get(0).alias("Cre"),
        pl.col("File name").alias("experiment"),
    )
    .select("Cre", "experiment")
)

de_results.join(other=more_metada, on="experiment").unique().write_csv(
    results_dir / "04_turboid_downstream_results/05_results/turboid_differential_expression.csv"
)

In [ ]:
df = pl.read_csv(
    results_dir
    / "04_turboid_downstream_results/05_results/turboid_differential_expression.csv"
)

print(set(df["comparison"]))
df = df.filter(pl.col("experiment").str.contains_any(EXPERIMENTS_TO_DROP).not_())



# Get max and min of non-infinite values
max_val = df.select(pl.col("log2_FC").filter(~pl.col("log2_FC").is_infinite()).max()).item()
min_val = df.select(pl.col("log2_FC").filter(~pl.col("log2_FC").is_infinite()).min()).item()

# Replace +inf with max, -inf with min
df = df.with_columns(
    pl.when(pl.col("log2_FC") == float('inf'))
    .then(max_val)
    .when(pl.col("log2_FC") == float('-inf'))
    .then(min_val)
    .otherwise(pl.col("log2_FC"))
    .alias("log2_FC")
)

print(df)


df = df.rename({"-log10_pval": "neg_log10_pval"}).unpivot(
    on=cs.by_name(["log2_FC", "Regulation", "neg_log10_pval"]),
    index = ~cs.by_name(["log2_FC", "Regulation", "neg_log10_pval"]),
)

In [ ]:


df = (
    s1_df.unique()
    .select(["experiment", "diet_duration", "type", "cre"])
    .with_columns(
        pl.when(pl.col("diet_duration") == "NA")
        .then(None)
        .otherwise(pl.col("diet_duration"))
        .alias("diet_duration")
    )
    .join(other=df, on="experiment", how="right")
    .with_columns(
        pl.concat_str(
            pl.col("cre"),
            pl.col("type"),
            pl.col("variable"),
            pl.col("comparison"),
            pl.col("diet_duration"),
            pl.col("experiment"),
            separator="_",
            ignore_nulls=True,
        ).alias("experiment")
    )
    .filter(
        pl.col("experiment").str.contains("_cre").not_()
        | pl.col("experiment").str.contains("EV1_105"),
        pl.col("experiment").str.contains_any(["all_conditions", "fast vs. LPS"]).not_(),
    )
    .pivot(
        on="experiment",
        values=["value"],
        index="uniprot_id",
        sort_columns=True,
    )
)
df = df.select(
    "uniprot_id",
    *sorted([col for col in df.columns if col != "uniprot_id"], key=lambda x: x[::-1])
)

df = protein_metadata.join(other=df, on="uniprot_id", how="right")

df.write_excel(
    workbook=supplementary_data_dir / f"Data S3.xlsx",
    position=(2, 0),
    freeze_panes=(3, 0),
    table_style=f"Table Style Light 8",
    column_widths=150,
    column_formats={"published concentration in mouse plasma (pmol ml-1)":"0.0"}
)

## Data S4: Proteome pheome atlas assocations

Protein disease associations for proteins quantified in the mouse secretome. Associations were retrieved retrieved from the proteome phenome atlas. Associations are filtered to prevalent associations with Boneferroni-corrected p-value < 0.05, in chapters I, IX, and IV.

In [ ]:
data_s4 = (
    pl.read_csv(
        turboid_dir
        / "../Reference datasets/UKBB_plasma_proteome_atlas/ukbb_turboid_intersect_significant_associations_only.csv"
    )
    .filter(pl.col("disease_type") == "prevalent")
    .select(
        "Protein",
        "Protein_definition",
        "Disease_category",
        "Disease",
        "NB_individual",
        "NB_case",
        "P_value",
        "OR[95%CI]",
    )
    .filter(
        pl.col("Disease_category").is_in(
            [
                "Chapter I Certain infectious and parasitic diseases",
                "Chapter IX Diseases of the circulatory system",
                "Chapter IV Endocrine, nutritional and metabolic diseases",
            ]
        )
    )
)

data_s4.write_excel(
    workbook=supplementary_data_dir / f"Data S4.xlsx",
    position=(2, 0),
    freeze_panes=(3, 0),
    table_style=f"Table Style Light 8",
    column_widths=150,
)

In [ ]:
pl.read_csv(
    "/Users/henrysanford/Downloads/pharos data download/query results.csv"
).filter(
    pl.col("Symbol").is_in(
        [
            "SNCG",
            "LEP",
            "APOE",
            "SERPINF1",
            "LRG1",
            "PLTP",
            "LIFR",
            "CES1",
            "FKBP5",
            "H6PD",
        ]
    )
).write_csv("/Users/henrysanford/Downloads/pharos data download/idg_targets.csv")

# Metadata for PRIDE submission

Create organized metadata of all TurboID experiments

In [ ]:
import polars as pl
import polars.selectors as cs
from pathlib import Path

turboid_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript"
)

data_s1 = (
    pl.read_excel(
        turboid_dir / "Supplementary Data/Supplementary Data_for review/Data S1.xlsx",
        read_options={"header_row": 2},
    )
    .select(~cs.contains("number"))
    .select(~cs.contains("percent"))
)
metadata_col = (
    pl.read_excel(
        turboid_dir
        / "Mass-spectrometry-datasets/02_metadata/metadata_turbo_id 2025.xlsx",
        sheet_name="metadata_col",
    )
    .with_columns(
        pl.col("File name ")
        .str.split("processed_census-out_")
        .list.get(1)
        .alias("experiment")
    )
    # only include experiments used in the study
    .filter(pl.col("experiment").is_in(data_s1["experiment"]))
    .pivot(on="Channel name", index="experiment", values="Condition name ")
    .with_columns(
        pl.concat_str(
            pl.lit("census-out_"), pl.col("experiment"), pl.lit(".txt")
        ).alias("census_out_file")
    )
)

(turboid_dir / "Supplementary Data/PRIDE_submission").mkdir(exist_ok=True, parents=True)

data_s1.join(metadata_col, "experiment", "left").write_excel(
    workbook=turboid_dir / "Supplementary Data/PRIDE_submission" / "Metadata.xlsx",
    freeze_panes=(1, 0),
    table_style=f"Table Style Light 8",
    column_widths=150,
)

# Tissue abundance data

Making a nicely formatted table of cell-type specifc proteins and their abundance in our data

In [ ]:
data_dir = Path("/Users/henrysanford/dev/turboid-figures")

tissue_abundance = pl.read_csv(data_dir / "tissue_abundance_data.csv")

hpa_data = (
    pl.read_csv(data_dir / "human_protein_atlas_lymphoid_liver_adipose.csv")
    .rename({"synonyms": "human_gene_name"})
    .drop("key")
)

highlight_proteins = set(
    pl.read_csv(data_dir / "reference_data/abundance_highlight_proteins.csv")
    .filter(pl.col("Keep for final figure?").eq("YES"))
    .with_columns(
        pl.concat_str(
            pl.col("tissue_to_highlight_in"), pl.lit("_"), pl.col("alias")
        ).alias("key")
    )["key"]
)

grouping = ["uniprot_id", "alias", "tissue_name", "log2_mean_SI", "rank", "tissue"]
df = (
    tissue_abundance.select(
        [
            "uniprot_id",
            "alias",
            "tissue_name",
            "tissue",
            "Symbol_human_jackson_homology_db",
            "log2_mean_SI",
            "rank",
        ]
    )
    .with_columns(
        pl.col("Symbol_human_jackson_homology_db")
        .str.split(",")
        .alias("human_gene_name")
    )
    .drop("Symbol_human_jackson_homology_db")
    .explode("human_gene_name")
    .join(other=hpa_data, on=["tissue", "human_gene_name"], how="left")
    # .drop("tissue")
    .group_by(grouping)
    .agg(pl.all())
    .with_columns(
        cs.exclude(grouping)
        .list.eval(pl.element().cast(pl.String), parallel=True)
        .list.unique()
        .list.join("|")
    )
    # .filter(pl.col("alias").str.starts_with("IGHV"))
    .drop("hpa_category")
    .with_columns(
        pl.concat_str(pl.col("tissue"), pl.lit("_"), pl.col("alias")).alias("key")
    )
    .with_columns(
        pl.when(
            (pl.col("key").is_in(highlight_proteins))
            & (pl.col("RNA single cell type specificity").eq(pl.lit("")).not_())
        )
        .then(pl.lit("YES"))
        .otherwise(pl.lit(""))
        .alias("label_in_plot?")
    )
    .drop("tissue", "key")
    .filter(pl.col("RNA single cell type specificity").eq(pl.lit("")).not_())   
)

df.write_csv(data_dir / "Abundance_plot_data.csv")

# Check abundance ratios of plasma data

In the TurboID pipeline, tissue data is filtered using ROC on ratio to Cre- while plasma is not filtered

This notebook determines which plasma proteins would be filtered out with a Cre- ratio requirement of 1.15 (15% enrichment)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from pathlib import Path
import numpy as np
import math

roc_filtered_path = Path(
    "/Volumes/Users/hsanford/Documents/turbo_id_data/census_out_processing_results_plasma_only/results/7/05_results/01_tissue"
)
no_roc_path = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/03_results/04_turboid_downstream_results/05_results/02_serum"
)

aliases = pl.read_csv("/Users/henrysanford/dev/turboid-figures/reference_data/protein_aliases.csv", truncate_ragged_lines=True)

aliases  = dict(zip(aliases["protein"].to_list(), aliases["alias"].to_list()))

metadata_df = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/TurboID manuscript/Mass-spectrometry-datasets/02_metadata/conditions_metadata.csv"
).with_columns(
    pl.col("File name").str.split("census-out_").list.get(1).alias("experiment")
).filter(
    pl.col("file_type").eq("serum")
).select(["conditions","experiment"])
rows = []


fc_cutoff = 1.15
for experiment in metadata_df["experiment"]:
    exp_short = experiment.split("_",maxsplit=1)[1]
    print(exp_short)
    no_roc_df = pl.read_csv(
        no_roc_path / f"{experiment}/final_protein_table_{experiment}.csv"
    ).drop("").with_columns(pl.col("Entry Name").str.split("_").list.get(0).replace(aliases).alias("protein"))

    data = no_roc_df.select(
        cs.contains("(median_cre+/(median_cre-)"), "uniprot_id", "annotation","Entry Name","Mass"
    ).with_columns(
        pl.col("annotation").fill_null("other")
    ).unpivot(
        index=["uniprot_id", "annotation", "Entry Name", "Mass"]
    ).with_columns(
        pl.col("value").log(base = 2).alias("log2(cre+/cre-)"),
        pl.col("variable").str.split("_").list.get(0).alias("condition"),
        pl.col("Entry Name").str.split("_").list.get(0).replace(aliases).alias("protein"),
    )

    dropped_proteins = data.group_by(
        ["protein",]
    ).agg(
        pl.col("log2(cre+/cre-)").max()
    ).filter(pl.col("log2(cre+/cre-)") < math.log2(fc_cutoff))

    no_roc_df.filter(pl.col("protein").is_in(dropped_proteins["protein"]).not_()).write_csv(
        no_roc_path / f"{experiment}/final_protein_table_{experiment}_enrichment_filtered.csv"
    )
    
    rows.append({
        "experiment":exp_short,
        "proteins before FC cutoff filter": len(no_roc_df),
        "proteins after FC cutoff filter": len(no_roc_df) - len(dropped_proteins),
        "proteins filtered out": list(dropped_proteins["protein"]),
        "fc_cutoff": fc_cutoff,
        "percentage_dropped":(len(dropped_proteins) / len(no_roc_df)) * 100
    })
df = pl.from_dicts(rows)

fc_requirement = 1.15
for exp in set(df["experiment"]):
    print(exp)
    f = list(df.filter(pl.col("fc_cutoff").eq(fc_requirement), pl.col("experiment").eq(exp))["proteins filtered out"])[0]
    print(", ".join(f))

sns.lineplot(data = df.drop("proteins filtered out"), x = "fc_cutoff", y = "percentage_dropped", hue = "experiment",palette="Set2" )
sns.set_style("whitegrid")
plt.ylabel("% of proteins dropped")
plt.xlabel("cre+/cre- FC requirement")
plt.legend(loc= "upper right")
plt.axvline(x=fc_requirement, color='red', linestyle='-')